## 3A.Dropout

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, models
import pandas as pd
import importlib
import src.utils as utils
importlib.reload(utils)

from src.utils import train_and_evaluate





In [ ]:
# Define a medium complexity CNN model with dropout layers
def Medium_model_with_dropout(dropout_rate=0.0):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Dropout(dropout_rate),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Dropout(dropout_rate),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),

        layers.Dropout(dropout_rate),

        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
# Define different dropout rates to test
configs = {
    "D0": 0.0,
    "D1": 0.25,
    "D2": 0.5
}

results = []
histories = {}

for name, rate in configs.items():
    model = Medium_model_with_dropout(rate)

    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\nTraining {name}...")

    history, acc, loss, t = train_and_evaluate(
        model,
        x_train_C, y_train,
        x_val_C, y_val,
        x_test_C, y_test,
        epochs=30
    )

    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    gap = train_acc - val_acc

    results.append((name, train_acc, val_acc, acc, gap))
    histories[name] = history

In [ ]:
# Display results in a DataFrame
df = pd.DataFrame(results, columns=[
    "Model", "Train Acc", "Val Acc", "Test Acc", "Overfit Gap"
])
df

In [ ]:
# Plot validation accuracy curves
plt.figure()
for name, h in histories.items():
    plt.plot(h.history['val_accuracy'], label=name)

plt.title("Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
# Plot training accuracy curves
plt.figure()
for name, h in histories.items():
    plt.plot(h.history['accuracy'], label=name)

plt.title("Train Accuracy")
plt.legend()
plt.show()

## 3B.EarlyStopping

In [ ]:
from tensorflow.keras import callbacks
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
import pandas as pd
import importlib
import src.utils as utils
importlib.reload(utils)

from src.utils import train_and_evaluate
from src.utils import BaselineCNN
from src.utils import run_Experiment_C






In [ ]:
es2 = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

es3 = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
experiments = {
    "ES0": None,
    "ES2": es2,
    "ES3": es3
}

results_es = []
hist_es = {}

for name, cb in experiments.items():
    model = Medium_model_with_dropout(0.0)

    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\nTraining {name}...")

    history = model.fit(
        x_train_C, to_categorical(y_train,10),
        validation_data=(x_val_C, to_categorical(y_val,10)),
        epochs=50,
        batch_size=128,
        callbacks=[cb] if cb else [],
        verbose=0
    )

    stopped_epoch = len(history.history['loss'])
    best_val_loss = min(history.history['val_loss'])

    test_loss, test_acc = model.evaluate(
        x_test_C, to_categorical(y_test,10), verbose=0
    )

    results_es.append((name, stopped_epoch, best_val_loss, test_acc))
    hist_es[name] = history

In [ ]:
df_es = pd.DataFrame(results_es, columns=[
    "Experiment", "Stopped Epoch", "Best Val Loss", "Test Acc"
])
df_es

In [ ]:
plt.figure()

for name, h in hist_es.items():
    plt.plot(h.history['val_loss'], label=name)

    stop_epoch = len(h.history['val_loss'])
    plt.axvline(stop_epoch, linestyle='--')

plt.legend()
plt.title("Val Loss with Early Stopping")
plt.show()